In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QESEM: פונקציית Qiskit מאת Qedma
> **Note:** פונקציות Qiskit הן תכונה ניסיונית הזמינה רק למשתמשי IBM Quantum&reg; Premium Plan, Flex Plan ו-On-Prem (דרך IBM Quantum Platform API) Plan. הן נמצאות בשלב טרום-הוצאה לשוק וכפופות לשינויים.

## סקירה כללית
בעוד שיחידות עיבוד קוונטיות השתפרו מאוד בשנים האחרונות, שגיאות הנובעות מרעש ופגמים בחומרה הקיימת נותרות אתגר מרכזי עבור מפתחי אלגוריתמים קוונטיים. ככל שהתחום מתקרב לחישובים קוונטיים בקנה מידה של תועלת שלא ניתן לאמת קלאסית, פתרונות לביטול רעש בדיוק מובטח הופכים חשובים יותר ויותר. כדי להתגבר על אתגר זה, Qedma פיתחה את Quantum Error Suppression and Error Mitigation (QESEM), המשולב בצורה חלקה ב-IBM Quantum Platform כ-[פונקציית Qiskit](/guides/functions).

עם QESEM, משתמשים יכולים להריץ את ה-Circuits הקוונטיים שלהם על QPUs רועשים ולקבל תוצאות מדויקות ללא שגיאות עם תקורות זמן QPU יעילות ביותר, קרוב לגבולות היסודיים. כדי להשיג זאת, QESEM מנצל חבילה של שיטות קנייניות שפותחו על ידי Qedma לאפיון והפחתת שגיאות. טכניקות הפחתת שגיאות כוללות אופטימיזציית Gate, Transpilation מודע לרעש, דיכוי שגיאות (ES) ומיתון שגיאות לא מוטה (EM). עם שילוב שיטות מבוססות-אפיון אלו, משתמשים יכולים להשיג תוצאות אמינות ללא שגיאות עבור Circuits קוונטיים גדולים וגנריים, ולפתוח יישומים שלא ניתן להשיגם אחרת.

לתיאור מלא של הרכיבים הבסיסיים, כמו גם הדגמה בקנה מידה של תועלת, עיין במאמר [Reliable high-accuracy error mitigation for utility-scale quantum circuits.](https://arxiv.org/abs/2508.10997)
## תיאור
תוכל להשתמש בפונקציית QESEM של Qedma כדי להעריך ולבצע בקלות את ה-Circuits שלך עם דיכוי ומיתון שגיאות, תוך השגת נפחי Circuit גדולים יותר ודיוק גבוה יותר. לשימוש ב-QESEM, עליך לספק Circuit קוונטי, קבוצה של אוברווביילים למדידה, דיוק סטטיסטי יעד לכל אוברוובייל, ו-QPU שנבחר. לפני הרצת ה-Circuit לדיוק היעד, תוכל להעריך את זמן ה-QPU הנדרש בהתבסס על חישוב אנליטי שאינו מחייב ביצוע Circuit. לאחר שתהיה מרוצה מהערכת זמן ה-QPU, תוכל להריץ את ה-Circuit עם QESEM.

כאשר אתה מריץ Circuit, QESEM מבצע פרוטוקול אפיון מכשיר המותאם ל-Circuit שלך, ומניב מודל רעש אמין עבור השגיאות המתרחשות ב-Circuit. בהתבסס על האפיון, QESEM מיישם תחילה Transpilation מודע לרעש כדי למפות את ה-Circuit הקלט על קבוצת qubits ו-Gates פיזיים, אשר ממזערת את הרעש המשפיע על האוברוובייל היעד. אלה כוללים את ה-Gates הזמינים ילידים (CX/CZ על מכשירי IBM&reg;), כמו גם Gates נוספות שאופטמו על ידי QESEM, ויוצרים את מערך ה-Gate המורחב של QESEM. לאחר מכן QESEM מריץ קבוצה של Circuits ES ו-EM מבוססי-אפיון על ה-QPU ואוסף את תוצאות המדידה שלהם. אלה עוברים לאחר מכן עיבוד קלאסי כדי לספק ערך ציפייה לא מוטה ופס שגיאה לכל אוברוובייל, המתאים לדיוק המבוקש.

![סקירת Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
QESEM הוכח כמספק תוצאות בדיוק גבוה עבור מגוון יישומים קוונטיים ובנפחי Circuit הגדולים ביותר הניתנים להשגה כיום. QESEM מציע את התכונות הבאות הפונות למשתמש, המוצגות בסעיף הבנצ'מרקים להלן:
-	**דיוק מובטח:** QESEM מפיק הערכות לא מוטות לערכי ציפייה של אוברווביילים. שיטת ה-EM שלו מצוידת בערבויות תיאורטיות, אשר - יחד עם אפיון החתכים של Qedma - מבטיחות שהמיתון מתכנס לפלט Circuit ללא רעש עד לדיוק המוגדר על ידי המשתמש. בניגוד לשיטות EM היוריסטיות רבות הנוטות לשגיאות שיטתיות או הטיות, הדיוק המובטח של QESEM חיוני להבטחת תוצאות אמינות ב-Circuits ואוברווביילים גנריים.
-	**מדרגיות ל-QPUs גדולים:** זמן ה-QPU של QESEM תלוי בנפחי Circuit, אך אינו תלוי במספר ה-Qubits. Qedma הדגים את QESEM על המכשירים הקוונטיים הגדולים ביותר הזמינים כיום, כולל IBM Quantum Eagle בן 127 Qubit ומכשירי Heron בני 133 Qubit.
-	**אגנוסטי ליישום:** QESEM הוכח על מגוון יישומים, כולל סימולציית Hamiltonian, VQE, QAOA ואמידת משרעת. משתמשים יכולים להזין כל Circuit קוונטי ואוברוובייל למדידה, ולקבל תוצאות מדויקות ללא שגיאות. המגבלות היחידות נקבעות על ידי מפרטי החומרה וזמן ה-QPU המוקצה, הקובעים את נפחי ה-Circuit הנגישים ואת דיוק הפלט. בניגוד לכך, פתרונות הפחתת שגיאות רבים ספציפיים ליישום או כוללים היוריסטיקה בלתי נשלטת, מה שהופך אותם לבלתי ישימים ל-Circuits ויישומים קוונטיים גנריים.
-  **מערך Gate מורחב:** QESEM תומך ב-Gates עם זוויות שבריות, ומספק Gates $Rzz(\theta)$ אופטמיזציה על ידי Qedma על מכשירי IBM Quantum Eagle. מערך Gate מורחב זה מאפשר קומפילציה יעילה יותר ופותח נפחי Circuit גדולים יותר פי 2 בהשוואה לקומפילציית CX/CZ ברירת מחדל.
-	**אוברווביילים מרובי בסיסים:** QESEM תומך באוברווביילי קלט המורכבים ממחרוזות Pauli רבות שאינן מתחלפות, כגון Hamiltonians גנריים. בחירת בסיסי המדידה ואופטימיזציית הקצאת משאבי ה-QPU (shots ו-Circuits) מבוצעת אז באופן אוטומטי על ידי QESEM כדי למזער את זמן ה-QPU הנדרש לדיוק המבוקש. אופטימיזציה זו, המתחשבת בנאמנויות חומרה וקצבי ביצוע, מאפשרת לך להריץ Circuits עמוקים יותר ולהשיג דיוק גבוה יותר.
## בנצ'מרקים
QESEM נבדק על מגוון רחב של מקרי שימוש ויישומים. הדוגמאות הבאות יכולות לסייע לך להעריך אילו סוגי עומסי עבודה ניתן להריץ עם QESEM.

מדד ביצועים מרכזי לכימות קושי מיתון השגיאות וסימולציה קלאסית עבור Circuit ואוברוובייל נתונים הוא **נפח פעיל**: מספר ה-Gate CNOT המשפיעים על האוברוובייל ב-Circuit. הנפח הפעיל תלוי בעומק ורוחב ה-Circuit, במשקל האוברוובייל ובמבנה ה-Circuit, הקובע את חרוט האור של האוברוובייל. לפרטים נוספים, ראה את ההרצאה מ-[IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM מספק ערך גדול במיוחד במשטר הנפח הגבוה, ומספק תוצאות אמינות ל-Circuits ואוברווביילים גנריים.

![נפח פעיל](../docs/images/guides/qedma-qesem/active_volume.svg)

| יישום    | מספר Qubits | מכשיר | תיאור Circuit | דיוק | זמן כולל | שימוש Runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE circuit  | 8              | Eagle (r3) | 21 שכבות סה"כ, 9 בסיסי מדידה, שרשרת 1D                    | 98%      | 35 דקות      | 14 דקות         |
| Kicked Ising   | 28              | Eagle (r3) | 3 שכבות ייחודיות x 3 צעדים, טופולוגיית heavy-hex דו-ממדית                      | 97%     | 22 דקות      | 4 דקות          |
| Kicked Ising   | 28              | Eagle (r3) | 3 שכבות ייחודיות x 8 צעדים, טופולוגיית heavy-hex דו-ממדית                     | 97%      | 116 דקות      | 23 דקות          |
| סימולציית Hamiltonian Trotterized   | 40  | Eagle (r3)            | 2 שכבות ייחודיות x 10 צעדי Trotter, שרשרת 1D                    | 97%      | 3 שעות     | 25 דקות         |
| סימולציית Hamiltonian Trotterized   | 119  | Eagle (r3)           | 3 שכבות ייחודיות x 9 צעדי Trotter, טופולוגיית heavy-hex דו-ממדית                    | 95%      | 6.5 שעות     | 45 דקות         |
| Kicked Ising  | 136             | Heron (r2) | 3 שכבות ייחודיות x 15 צעדים, טופולוגיית heavy-hex דו-ממדית                    | 99%      | 52 דקות             | 9 דקות           |

הדיוק נמדד כאן ביחס לערך האידיאלי של האוברוובייל: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, כאשר '$\epsilon$' הוא הדיוק האבסולוטי של המיתון (שנקבע על ידי קלט המשתמש), ו-$\langle O \rangle_{ideal}$ הוא האוברוובייל ב-Circuit ללא רעש.
'שימוש Runtime' מודד את השימוש של הבנצ'מרק במצב batch (סכום על שימוש של עבודות בודדות), ואילו 'זמן כולל' מודד שימוש במצב session (זמן קיר הניסוי), הכולל זמני חישוב ותקשורת קלאסיים נוספים. QESEM זמין לביצוע בשני המצבים, כך שמשתמשים יכולים לעשות את השימוש הטוב ביותר במשאביהם הזמינים.

ה-Circuits של Kicked Ising בן 28 Qubit מדמים את Discrete Time Quasicrystal שנחקר על ידי Shinjo et al. (ראה [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) ו-[Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) על שלושה לולאות מחוברות של ibm_kawasaki. פרמטרי ה-Circuit שנלקחו כאן הם $(\theta_x, \theta_z) = (0.9 \pi, 0)$, עם מצב ראשוני פרומגנטי $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. האוברוובייל הנמדד הוא הערך המוחלט של המגנטיזציה $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. ניסוי Kicked Ising בקנה מידה של תועלת הופעל על 136 ה-Qubits הטובים ביותר של ibm_fez; בנצ'מרק ספציפי זה הופעל בזווית Clifford $(\theta_x, \theta_z) = (\pi, 0)$, שבה הנפח הפעיל גדל לאט עם עומק ה-Circuit, מה שמאפשר - יחד עם נאמנויות מכשיר גבוהות - דיוק גבוה בזמן ריצה קצר.

ה-Circuits של סימולציית Hamiltonian Trotterized הם עבור מודל Transverse-Field Ising בזוויות שבריות: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ ו-$(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ בהתאמה (ראה [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). ה-Circuit בקנה מידה של תועלת הופעל על 119 ה-Qubits הטובים ביותר של ibm_brisbane, ואילו ניסוי ה-40 Qubit הופעל על השרשרת הטובה הזמינה. הדיוק מדווח עבור המגנטיזציה; תוצאות בדיוק גבוה הושגו גם עבור אוברווביילים במשקל גבוה יותר.

ה-Circuit של VQE פותח יחד עם חוקרים ממרכז לטכנולוגיה וישומים קוונטיים ב-Deutsches Elektronen-Synchrotron (DESY). האוברוובייל היעד כאן היה Hamiltonian המורכב ממספר רב של מחרוזות Pauli שאינן מתחלפות, המדגיש את הביצועים האופטימיזציים של QESEM עבור אוברווביילים מרובי בסיסים. המיתון הוחל על ansatz שעבר אופטימיזציה קלאסית; למרות שתוצאות אלה עדיין לא פורסמו, תוצאות באותה איכות יתקבלו עבור Circuits שונים עם מאפיינים מבניים דומים.
## תחילת עבודה
אמת את זהותך באמצעות [מפתח ה-API של IBM Quantum Platform](http://quantum.cloud.ibm.com/), ובחר את פונקציית Qiskit של QESEM כדלקמן. (קטע קוד זה מניח שכבר [שמרת את החשבון שלך](/guides/functions#install-qiskit-functions-catalog-client) בסביבה המקומית שלך.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## דוגמה
כדי להתחיל, נסה דוגמה בסיסית זו להערכת זמן ה-QPU הנדרש להרצת QESEM עבור `pub` נתון:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

הדוגמה הבאה מבצעת עבודת QESEM:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

תוכל להשתמש בממשקי ה-API המוכרים של Qiskit Serverless כדי לבדוק את סטטוס עומס העבודה של פונקציית Qiskit שלך או להחזיר תוצאות:

In [ ]:
print(sample_job.status())
result = sample_job.result()

## פרמטרי הפונקציה
| שם |  סוג | תיאור | נדרש | ברירת מחדל |  דוגמה |
| -----| ------| ------------| -------- | ------- | -------- |
| `pubs` | [EstimatorPubLike](/guides/primitive-input-output) |זהו הקלט הראשי. ה-`Pub` מכיל 2-4 רכיבים: Circuit, אוברוובייל אחד או יותר, 0 או קבוצה אחת של ערכי פרמטרים, ודיוק אופציונלי. אם לא צוין דיוק, ייעשה שימוש ב-`default_precision` מתוך `options`|  כן|  N/A |  `[(circuit, [obs1,obs2,obs3], parameter_values, 0.03)]`  |
| `backend_name`| string|שם ה-Backend לשימוש |לא | QESEM יבחר את המכשיר הפחות עסוק המדווח על ידי IBM| `"ibm_fez"`|
| `instance` | string|  שם משאב הענן (CRN) של ה-instance לשימוש בפורמט זה |  לא |  N/A | `"CRN"`  |
| `options` | dictionary |אפשרויות קלט. ראה את סעיף **אפשרויות** לפרטים נוספים. | לא |  ראה את סעיף **אפשרויות** לפרטים.    |  `{ default_precision = 0.03, "max_execution_time" = 3600, "transpilation_level" = 0}`  |

### אפשרויות
| אפשרות |  בחירות | תיאור | ברירת מחדל |
| -----| -----------| -------- | ------- |
| `estimate_time_only` |  `"analytical"`  / `"empirical"` / None  | כאשר מוגדר, עבודת QESEM תחשב רק את הערכת זמן ה-QPU. ראה את התיאור הבא לפרטים נוספים. כאשר מוגדר ל-None, ה-Circuit יבוצע עם QESEM| None |
|`default_precision` | 0 < float | יוחל על `pubs` שאין להם דיוק. הדיוק מציין את השגיאה המקובלת על ערכי הציפייה של האוברווביילים בערך מוחלט. כלומר, זמן ה-QPU לביצוע המיתון ייקבע כך שיספק ערכי פלט לכל האוברווביילים שעניין בהם הנופלים בתוך רווח בטחון `1σ` מהדיוק היעד. אם מסופקים מספר אוברווביילים, המיתון ירוץ עד להשגת דיוק היעד עבור כל אחד מהאוברווביילים שהוזנו. | 0.02|
|`max_execution_time` | 0 < integer < 28,800 (8 שעות)| מאפשר לך להגביל את זמן ה-QPU, המוגדר בשניות, לשימוש עבור כל תהליך QESEM. מצא פרטים נוספים להלן.| 3,600 (שעה אחת)|
| `transpilation_level` | 0 / 1 | ראה תיאור להלן | 1|
| `execution_mode` | `"session"` / `"batch"` | ראה תיאור הבא | "batch"|

> **Caution:** הערכת זמן ה-QPU משתנה מ-Backend אחד לאחר. לכן, בעת ביצוע פונקציית QESEM, ודא שאתה מריץ אותה על אותו Backend שנבחר בעת קבלת הערכת זמן ה-QPU.

> **Note:** QESEM יסיים את הרצתו כאשר יגיע לדיוק היעד או כאשר יגיע ל-`max_execution_time`, לפי המוקדם מביניהם.

- `estimate_time_only` - דגל זה מאפשר למשתמשים לקבל הערכה לזמן ה-QPU הנדרש לביצוע ה-Circuit עם QESEM.
    - אם מוגדר ל-`"analytical"`, חסם עליון של זמן ה-QPU מחושב ללא צריכת כל שימוש ב-QPU. הערכה זו בעלת רזולוציה של 30 דקות (לדוגמה, 30 דקות, 60 דקות, 90 דקות, וכן הלאה). היא פסימית בדרך כלל, וניתן להשיגה רק עבור אוברווביילים בודדים של Pauli או סכומי Pauli ללא תמיכות חופפות (לדוגמה, Z0+Z1). היא שימושית בעיקר להשוואת רמות המורכבות של פרמטרים שונים שסיפק המשתמש (Circuit, דיוק וכן הלאה).
    - לקבלת הערכת זמן QPU מדויקת יותר, הגדר דגל זה ל-`"empirical"`. למרות שאפשרות זו דורשת הרצה של מספר קטן של Circuits, היא מספקת הערכת זמן QPU מדויקת משמעותית. הערכה זו בעלת רזולוציה של 5 דקות (לדוגמה, 20 דקות, 25 דקות, 30 דקות, וכן הלאה). המשתמש רשאי לבחור להריץ את הערכת הזמן האמפירית במצב batch או session. לפרטים נוספים, ראה את תיאור `execution_mode`. לדוגמה, במצב batch, הערכת זמן אמפירית תצרוך פחות מ-10 דקות של זמן QPU.

- `max_execution_time`: מאפשר לך להגביל את זמן ה-QPU, המוגדר בשניות, לשימוש עבור כל תהליך QESEM. מכיוון שזמן ה-QPU הסופי הנדרש להגעה לדיוק היעד נקבע באופן דינמי במהלך עבודת QESEM, פרמטר זה מאפשר לך להגביל את עלות הניסוי. אם זמן ה-QPU שנקבע דינמית קצר מהזמן שהוקצה על ידי המשתמש, פרמטר זה לא ישפיע על הניסוי. הפרמטר `max_execution_time` שימושי במיוחד במקרים שבהם הערכת הזמן האנליטית שסיפקה QESEM לפני תחילת העבודה פסימית מדי והמשתמש רוצה ליזום עבודת מיתון בכל זאת. לאחר הגעה למגבלת הזמן, QESEM מפסיק לשלוח Circuits חדשים. Circuits שכבר נשלחו ממשיכים לרוץ (כך שהזמן הכולל עשוי לחרוג מהמגבלה בעד 30 דקות), והמשתמש מקבל את התוצאות המעובדות מה-Circuits שרצו עד לאותה נקודה. אם ברצונך להחיל מגבלת זמן QPU קצרה מהערכת הזמן האנליטית, התייעץ עם Qedma לקבלת הערכה לדיוק הניתן להשגה בתוך מגבלת הזמן.

- `transpilation_level`: לאחר שליחת Circuit ל-QESEM, הוא מכין אוטומטית מספר Transpilations חלופיים של ה-Circuit ובוחר את זה שממזער את זמן ה-QPU. לדוגמה, Transpilations חלופיים עשויים לנצל Gates RZZ שבריים אופטמיזציה על ידי Qedma כדי להפחית את עומק ה-Circuit. כמובן, כל ה-Transpilations שקולים לקלט ה-Circuit, מבחינת פלטם האידיאלי. לשליטה רבה יותר על Transpilation ה-Circuit, הגדר את רמת ה-Transpilation ב-`options`. בעוד ש-`"transpilation_level": 1` מתאים להתנהגות ברירת המחדל שתוארה לעיל, `"transpilation_level": 0` כולל רק שינויים מינימליים נדרשים ל-Circuit המקורי; לדוגמה, 'layerification' - ארגון פעולות ה-Circuit ל'שכבות' של Gates דו-Qubit סימולטניות. שים לב שמיפוי חומרה אוטומטי על Qubits בעלי נאמנות גבוהה מוחל בכל מקרה.

| transpilation_level | תיאור |
|:-:|:--|
| `1` | Transpilation ברירת מחדל של QESEM. מכין מספר Transpilations חלופיים ובוחר את זה שממזער את זמן ה-QPU. Barriers עשויים להשתנות בשלב ה-layerification. |
| `0` | Transpilation מינימלי: ה-Circuit המוקל יהיה דומה מבחינה מבנית לקלט ה-Circuit. Circuits המסופקים ברמה 0 צריכים להתאים לקישוריות המכשיר ולהיות מוגדרים במונחים של ה-Gates הבאות: CX, Rzz(α), ו-Gates חד-Qubit סטנדרטיות (U, x, sx, rz, וכן הלאה). Barriers יכובדו בשלב ה-layerification. |

- `execution_mode` - המשתמש יכול לבחור להריץ את עבודת QESEM ב-[IBM session](/guides/execution-modes#session-mode) ייעודי או פרוס על פני מספר [IBM batches](/guides/execution-modes#batch-mode):
    -   **מצב Session**: Sessions יקרים יותר אך מביאים לזמן-עד-תוצאה קצר יותר. לאחר שה-Session מתחיל, ה-QPU שמור באופן בלעדי לעבודת QESEM. חישוב השימוש כולל את הזמן המושקע בביצוע ה-QPU וחישובים הקלאסיים הנלווים (המבוצעים על ידי QESEM ו-IBM). פונקציית Qiskit של QESEM דואגת ליצירת ה-Session וסגירתו באופן אוטומטי. למשתמשים עם גישה בלתי מוגבלת ל-QPUs (לדוגמה, הגדרות on-premises), מומלץ להשתמש במצב session לביצוע QESEM מהיר יותר.
    -   **מצב Batch**: במצב batch, ה-QPU משוחרר במהלך חישובים קלאסיים, מה שמוביל לשימוש נמוך יותר ב-QPU. מכיוון שעבודות batch בדרך כלל נפרסות על פני תקופה ארוכה יותר, קיים סיכון גדול יותר לסחיפות חומרה; QESEM משלב אמצעים לאיתור ופיצוי על סחיפות, תוך שמירה על אמינות בריצות ממושכות.

> **Note:** פעולות Barrier משמשות בדרך כלל לציון שכבות של Gates דו-Qubit ב-Circuits קוונטיים. ברמה 0, QESEM שומר על השכבות שצוינו על ידי ה-Barriers. ברמה 1, השכבות שצוינו על ידי ה-Barriers נחשבות כחלופת Transpilation אחת בעת מיזעור זמן ה-QPU.
### פלטים
הפלט של פונקציית Circuit הוא [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), המכיל שני שדות:

- אובייקט [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) אחד. ניתן לאנדקס אותו ישירות מה-`PrimitiveResult`.

- מטא-דאטה ברמת עבודה.

כל `PubResult` מכיל שדה `data` ושדה `metadata`.

- השדה `data` מכיל לפחות מערך של ערכי ציפייה (`PubResult.data.evs`) ומערך של שגיאות סטנדרטיות (`PubResult.data.stds`). הוא יכול לכלול גם נתונים נוספים, בהתאם לאפשרויות שנעשה בהן שימוש.

- השדה `metadata` מכיל מטא-דאטה ברמת PUB (`PubResult.metadata`).

קטע הקוד הבא מתאר כיצד לאחזר את הערכת זמן ה-QPU (כאשר `estimate_time_only` מוגדר):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

קטע הקוד הבא מדגים כיצד לאחזר את תוצאות המיתון (כאשר `estimate_time_only` אינו מוגדר) ומדדי ביצוע. אלה מכילים נתונים חיוניים המאפשרים הבנה עמוקה יותר של כיצד פרמטרים שונים משפיעים על ביצוע QESEM. זה עשוי להיות רלוונטי גם בעת כתיבת מאמר המבוסס על המחקר שלך.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## אחזור הודעות שגיאה
אם סטטוס עומס העבודה שלך הוא ERROR, השתמש ב-job.result() כדי לאחזר את הודעת השגיאה כדלקמן:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem)
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199.](https://arxiv.org/pdf/2507.01199)
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997.](https://arxiv.org/pdf/2508.10997)


</Admonition>